In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

current_path = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [current_path, *current_path.parents] if (path / "src").exists()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root containing 'src'.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.cleaner import clean_dataframe
from src.loader import load_datasets
from src.merger import merge_multiple_datasets
from src.validator import validate_dataset

DATA_DIR = PROJECT_ROOT / "episodios" / "001_embarazo_adolescente" / "data" / "raw"


In [ ]:
dataset_paths = {
    "embarazo": DATA_DIR / "embarazo_adolescentes.csv",
    "educacion": DATA_DIR / "educacion.csv",
    "demografia": DATA_DIR / "demografia.csv",
    "condiciones": DATA_DIR / "condiciones_socieconomicas.csv",
    "acceso": DATA_DIR / "acceso_a_informacion.csv",
    "municipios": DATA_DIR / "municipios.csv",
}

datasets = load_datasets(dataset_paths)

list(datasets)


In [ ]:
inventory = []

for name, df in datasets.items():
    inventory.append(
        {
            "dataset": name,
            "rows": len(df),
            "columns": len(df.columns),
            "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        }
    )

inventory = pd.DataFrame(inventory)
display(inventory)


In [ ]:
cleaned_datasets = {}
validation_reports = {}

for name, df in datasets.items():
    cleaned = clean_dataframe(df)
    cleaned_datasets[name] = cleaned
    validation_reports[name] = validate_dataset(cleaned, dataset_name=f"{name}.csv")
    print(validation_reports[name]["report"])
    print()


In [ ]:
master_df = merge_multiple_datasets(cleaned_datasets, key="DIVIPOLA")
master_df.head()
